## 2.3 图像预处理 - Compose 与训练集 & 验证集 transforms 设计

#### 1. 为什么要学习这一节

##### 1.1 这一节要解决什么问题
前一小节中，我们已经分别学习了这些增强 API：
* RandomResizedCrop
* RandomHorizontalFlip
* RandomRotation
* ColorJitter
* ToTensor()
* 添加噪声

但是在真实项目里，我们不可能每次都手动写成这样：
``` python
img = random_crop(img)
img = random_flip(img)
img = random_rotate(img)
img = color_jitter(img)
img = to_tensor(img)
```

如果数据集里有成千上万张图片，每次都手动一行一行调用会非常麻烦。

所以我们需要一种方式，把多个预处理步骤打包成一个完整流程，这就是 Compose 的作用。


##### 1.2 这一节的三个重点
这一小节需要真正搞清楚三个问题：
* 为什么很多 torchvision.transforms 先处理的是 PIL.Image
* Compose 到底是什么，它解决了什么问题
* 为什么训练集和验证集 / 测试集要设计不同的 transforms

##### 1.3 这一节和前面“统一输入格式”的关系
这里特别容易产生一个误区：

前面我们说图像最后要统一成适合模型输入的格式，

那为什么这一节又会提到 PIL.Image？

答案是：
* 统一输入格式 说的是“最终喂给模型之前”
* PIL.Image 说的是“在很多增强 API 中间处理阶段的常见格式”

所以这两个并不冲突。

实际流程经常是：

`PIL.Image → 数据增强 → ToTensor() → Normalize() → 模型输入`

#### 2. PIL Image 在 transforms 中的地位

##### 2.1 什么是 PIL Image
PIL 是 Python 中一个经典的图像处理库，后来常见使用的是它的分支 Pillow。

我们平时说的 PIL.Image，本质上就是：

Pillow 库中的图像对象。

例如：
``` python
from PIL import Image

img = Image.open("cat.jpg")
print(type(img))
```

输出通常类似：

`<class 'PIL.JpegImagePlugin.JpegImageFile'>`

或者可以理解成 PIL.Image 类型的对象。

##### 2.2 为什么 torchvision 很多增强 API 喜欢处理 PIL Image
很多经典的 torchvision.transforms 图像增强操作，最初就是围绕 PIL.Image 使用场景设计的，比如：
* 裁剪 Crop
* 翻转 Flip
* 旋转 Rotation
* 颜色抖动 ColorJitter
* Resize

因为这些操作本质上更接近“图像层面”的处理，而不是“张量数值计算层面”的处理。

所以在传统写法中，经常是：
* 先用 Image.open() 读图
* 得到 PIL.Image
* 对 PIL.Image 做增强
* 最后再 ToTensor()

##### 2.3 这和 NumPy / Tensor 冲突吗
不冲突。

图像在 Python / PyTorch 工作流中常见有三种表现形式：

**（1）NumPy ndarray**

适合：
* 查看图像数组本质
* 手动做数值处理
* 配合 matplotlib / opencv

**（2）PIL.Image**

适合：
* 图像读写
* 调用很多 torchvision.transforms 的经典增强 API

**（3）torch.Tensor**

适合：
* 送入神经网络
* GPU 运算
* 自动求导
* 模型训练

📌 所以你可以把它们理解成三个不同阶段常用的“图像载体”。

##### 2.3 为什么前面我们还专门学了 NumPy
因为 NumPy 负责帮你理解图像的本质：
* 图像就是数组
* 有 shape
* 有通道
* 有 dtype
* 有数值范围

而 PIL.Image 更像是：
* 调用现成图像增强 API 时的方便格式

而 Tensor 则是：
* 最终进入深度学习模型的格式

所以三者是分工不同，不是互相替代。

##### 2.4 一个标准的思维顺序
建立这样一个理解链条：
* 图像本质上是数组（NumPy 思维）
* 图像增强常先基于 PIL.Image 做（transforms 思维）
* 模型最终接收的是 Tensor（PyTorch 思维）

#### 3. Compose 是什么

##### 3.1 Compose 的定义
Compose 是 torchvision.transforms 中用来组合多个预处理 / 增强操作的工具。

它可以把多个 transform 按顺序串起来，形成一个完整流程。

例如你可以理解成：
* 第一步做 Resize
* 第二步做 Flip
* 第三步做 ToTensor
* 第四步做 Normalize

把这些步骤打包成一个总变换对象。

##### 3.2 为什么需要 Compose
如果不用 Compose，你就得手动一行一行调用：
``` python
img = resize(img)
img = flip(img)
img = to_tensor(img)
img = normalize(img)
```
这样写当然可以，但会有几个问题：
* 代码不够整洁
* 容易漏步骤
* 不方便复用
* 不方便直接交给 Dataset 使用

而 Compose 可以把整个流程封装起来。

##### 3.3 Compose 的核心思想
Compose 的核心思想非常简单：

把多个小变换，拼成一个大变换。

这样你后面只需要写：

`img = transform(img)`

它就会自动按顺序完成所有步骤。

#### 4. Compose 的基本语法

##### 4.1 基本写法
Compose 的典型写法如下：
``` python
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    # 其他增强操作 ...... 
    transforms.ToTensor()
])
```
这里的意思是：
* 先 Resize
* 再进行其他增强操作
* 再 ToTensor

##### 4.2 使用方式
定义好之后，直接像函数一样调用：

`img = transform(img)`

这时 img 会自动按顺序经过前面定义的所有操作。

##### 4.3 顺序为什么重要
Compose 中各个步骤的顺序非常重要。⭐

因为前一个操作的输出，会作为后一个操作的输入。

例如：
* Resize, Corp, Flip, Rotation 一般处理 PIL.Image
* ToTensor() 之后输出 Tensor
* Normalize() 一般处理 Tensor

所以顺序通常要写成：
* 先 PIL.Image 相关操作
* 再 ToTensor()
* 再 Tensor 相关操作

#### 5. ToTensor() 操作

##### 5.1 oTensor() 做了什么
ToTensor() 的主要作用是：
1. 把 PIL.Image 或 NumPy 图像转成 torch.Tensor，包括dtype
2. 把像素值从 0~255 缩放到 0~1，完成归一化
3. 调整通道顺序为 PyTorch 更常见的形式

例如：
* 原来 PIL.Image
* 转后变成 Tensor，通常 shape 类似 (C, H, W)

这样就不需要 2.1 图像预处理 - 统一输入格式 中的：
1. 修改dtype
2. 大小归一化
3. 修改通道顺序，增加通道维度个数

##### 5.2 为什么很多增强写在 ToTensor() 前面
因为很多传统增强操作更自然地作用在 PIL.Image 上，例如：
* Resize
* RandomCrop
* RandomHorizontalFlip
* RandomRotation
* ColorJitter

所以通常会先写这些，再写 ToTensor()。

##### 5.3 为什么 Normalize() 往往放在 ToTensor() 后面
因为 Normalize() 需要处理的是 Tensor 数值。

它一般写成：

`transforms.Normalize(mean=[...], std=[...])`

而这一步的前提是：
* 图像已经是 Tensor
* 通道数已经明确
* 像素值范围已经合理

##### 5.4 一个典型顺序
典型流程通常是：
* Resize / Crop / Flip / Rotation / ColorJitter
* ToTensor()
* Normalize()

#### 6. 训练集 transforms 应该怎么设计

##### 6.1 训练集的目标是什么
训练集的目标不是“公平评估”，而是：

让模型充分学习，尽量提高泛化能力。

所以训练集通常会加入：
* 基础预处理
* 随机增强

##### 6.2 训练集 transforms 的典型思路
训练集一般会包含两类步骤：

**（1）统一输入格式的步骤**
* Resize
* ToTensor
* Normalize

**（2）增强泛化能力的步骤**
* RandomResizedCrop
* RandomHorizontalFlip
* RandomRotation
* ColorJitter

所以训练集通常会“更丰富、更随机”。

##### 6.3 为什么训练集要用随机增强
因为模型不能只记住固定的训练图像。

如果每次训练时图片都有一点变化，模型就会被迫学习更稳定、更本质的特征。

##### 6.4 一个训练集 transforms 示例
``` python
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop((224, 224), scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
```

##### 6.5 这个设计体现了什么思想
这个训练集设计体现的是：
* 前半段：做随机增强
* 后半段：统一格式并准备模型输入

也就是说：

先让图片变得更多样，再让输出格式变得更标准。

#### 7. 验证集 / 测试集 transforms 应该怎么设计

##### 7.1 验证集 / 测试集的目标是什么
它们的目标不是帮助模型学习，而是：

公平、稳定地评估模型性能。

所以这里最重要的是：
* 输入稳定
* 结果可重复
* 评估公平

##### 7.2 为什么不能随便加随机增强
如果你对验证集 / 测试集也加上：
* 随机裁剪
* 随机翻转
* 随机旋转

那就会出现问题：
* 每次评估的输入都不一样
* 每次指标可能不同
* 无法稳定比较模型性能

所以一般不这样做。

##### 7.3 验证集 / 测试集通常只保留基础处理
* Resize
* ToTensor()
* Normalize()

这样每次评估的输入都是确定性的，更公平。

##### 7.4 一个验证集 / 测试集 transforms 示例
``` python
from torchvision import transforms

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
```

#### 8. 训练集与验证集 transforms 的对比理解
**1️⃣ 训练集更像“练习题”**

训练阶段的目标是让模型多练、多见、多适应变化。

所以训练集会加入各种合理扰动。

**2️⃣ 验证集 / 测试集更像“正式考试题”**

正式考试需要规则统一、题目稳定，不能每次随机改题。

所以评估集通常只做基础预处理。

**3️⃣ 一个非常重要的结论**

随机增强主要属于训练策略，不属于评估策略。

#### 9. Compose 在 Dataset 中为什么特别方便

##### 9.1 Dataset 需要什么
在 PyTorch 中，我们后面常常会在自定义数据集或现成数据集中这样写：

`dataset = SomeDataset(transform=train_transform)`

也就是说，Dataset 希望你传进去的是一个“总的 transform”。

##### 9.2 如果没有 Compose 会怎样

如果没有 Compose，你就得在 __getitem__() 里手动一行一行写很多步骤，代码会比较乱。

##### 9.3 有了 Compose 的好处
有了 Compose，你就可以：
* 把预处理流程单独定义好
* 训练集一套
* 验证集一套
* Dataset 里直接复用

这会让代码结构更清楚。

#### 10. 一个完整示例：训练集与验证集 transforms 设计

In [1]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size=(224, 224),
        scale=(0.08, 1.0),
        ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])